# Rainfall Prediction
Predicting daily rainfall amounts from Australian weather station data.

## Project Overview
Regression project using the Weather Australia dataset to predict rainfall amount.

## Learning Objectives
- Work with real weather station data
- Handle high missing-value rates
- Build and compare regression models

## Problem Statement
Predict the amount of rainfall (mm) given daily weather measurements.

## Why This Project Matters
Rainfall forecasting is critical for agriculture, flood management, and urban planning.

## Dataset Overview
| Feature | Value |
|---|---|
| **Source** | Local: weatherAUS.csv |
| **Target** | Rainfall (mm) |
| **Type** | Regression |
| **Features** | Temperature, humidity, wind, pressure, cloud cover |

## Dataset Source & License
Australian Bureau of Meteorology. Public domain weather observations.

## Environment Setup

In [1]:
import subprocess, sys
for pkg in ['catboost','lightgbm','xgboost','flaml','lazypredict']:
    try: __import__(pkg)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install',pkg,'-q'])

C:\Users\ahmad\AppData\Local\Programs\Python\Python313\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


## Imports

In [2]:
import os, json, warnings, numpy as np, pandas as pd, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
TEST_SIZE = 0.2
SAVE_DIR = os.path.dirname(os.path.abspath('__file__'))
os.makedirs(SAVE_DIR, exist_ok=True)

## Configuration

In [3]:
TARGET = 'Rainfall'
MAX_ROWS = 50000

## Dataset Loading

In [4]:
# Load local weatherAUS.csv
csv_path = os.path.join(SAVE_DIR, 'weatherAUS.csv')
if not os.path.exists(csv_path):
    raise FileNotFoundError(f'Dataset not found at {csv_path}. Please place weatherAUS.csv in the project folder.')
df = pd.read_csv(csv_path)
if len(df) > MAX_ROWS:
    df = df.sample(MAX_ROWS, random_state=SEED)
print(f'Shape: {df.shape}')
df.head()

Shape: (50000, 23)


,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
100721,2012-04-22,MountGambier,15.0,18.9,4.2,6.6,8.0,NNW,54.0,NNW,...,73.0,64.0,1005.2,1003.7,4.0,5.0,17.3,17.6,Yes,Yes
30234,2008-03-30,Sydney,13.1,26.8,0.0,4.6,10.9,NaN,NaN,W,...,61.0,22.0,1013.0,1009.0,0.0,1.0,16.9,25.9,No,No
68427,2011-12-10,Melbourne,19.0,29.0,NaN,11.0,5.6,N,59.0,N,...,50.0,38.0,1006.5,1003.4,NaN,NaN,24.2,27.2,NaN,NaN
28624,2013-03-27,Richmond,18.1,32.2,0.0,2.1,NaN,NE,30.0,NaN,...,99.0,51.0,1019.2,1014.6,NaN,NaN,20.9,31.6,No,No
31173,2010-10-25,Sydney,13.9,19.6,14.0,1.2,5.4,SSW,50.0,SW,...,90.0,64.0,NaN,1018.4,7.0,6.0,15.5,19.1,Yes,No


## Data Validation

In [5]:
print('Missing values:')
print(df.isnull().sum())
print(f'\nDuplicates: {df.duplicated().sum()}')
print(f'\nTarget stats:')
print(df[TARGET].describe())

Missing values:
Date                 0
Location             0
MinTemp            520
MaxTemp            446
Rainfall          1123
Evaporation      21604
Sunshine         24031
WindGustDir       3595
WindGustSpeed     3569
WindDir9am        3623
WindDir3pm        1457
WindSpeed9am       611
WindSpeed3pm      1061
Humidity9am        902
Humidity3pm       1528
Pressure9am       5181
Pressure3pm       5160
Cloud9am         19252
Cloud3pm         20429
Temp9am            607
Temp3pm           1225
RainToday         1123
RainTomorrow      1147
dtype: int64

Duplicates: 0

Target stats:
count    48877.000000
mean         2.345185
std          8.548860
min          0.000000
25%          0.000000
50%          0.000000
75%          0.800000
max        371.000000
Name: Rainfall, dtype: float64


## Exploratory Data Analysis

In [6]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
df[TARGET].clip(upper=df[TARGET].quantile(0.99)).hist(bins=50, ax=axes[0,0], edgecolor='black')
axes[0,0].set_title('Rainfall Distribution')
num_cols = df.select_dtypes('number').columns.tolist()
plot_cols = [c for c in num_cols if c != TARGET][:3]
for i, col in enumerate(plot_cols):
    ax = axes[(i+1)//2, (i+1)%2]
    ax.scatter(df[col].head(2000), df[TARGET].head(2000), alpha=0.3, s=3)
    ax.set_title(f'{col} vs {TARGET}')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'eda_plots.png'), dpi=100)
plt.show()

## Target Analysis

In [7]:
print(df[TARGET].describe())
print(f'\nSkewness: {df[TARGET].skew():.2f}')
print(f'Zero rainfall: {(df[TARGET]==0).sum()} ({(df[TARGET]==0).mean()*100:.1f}%)')

count    48877.000000
mean         2.345185
std          8.548860
min          0.000000
25%          0.000000
50%          0.000000
75%          0.800000
max        371.000000
Name: Rainfall, dtype: float64

Skewness: 10.91
Zero rainfall: 31269 (62.5%)


## Train/Test Split

In [8]:
# Drop non-features
drop_cols = [c for c in ['Date','Location','RainToday','RainTomorrow'] if c in df.columns]
df = df.drop(columns=drop_cols)

# Encode categoricals
cat_cols = df.select_dtypes(include='object').columns.tolist()
for c in cat_cols:
    df[c] = df[c].astype('category').cat.codes

# Fill missing
df = df.fillna(df.median(numeric_only=True))
df = df.replace([np.inf, -np.inf], np.nan).fillna(0)

X = df.drop(columns=[TARGET])
y = df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_SIZE, random_state=SEED)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')

Train: (40000, 18), Test: (10000, 18)


## Preprocessing
Dropped Date/Location, encoded wind direction categoricals, filled missing with median.

## Feature Engineering
Using raw meteorological features. Tree models handle interactions naturally.

## Baseline Model

In [9]:
bl = Ridge(alpha=1.0)
bl.fit(X_train, y_train)
bl_pred = bl.predict(X_test)
print(f'Baseline Ridge: R2={r2_score(y_test, bl_pred):.4f}  RMSE={root_mean_squared_error(y_test, bl_pred):.4f}  MAE={mean_absolute_error(y_test, bl_pred):.4f}')

Baseline Ridge: R2=0.1394  RMSE=7.2655  MAE=3.4033


## LazyPredict Benchmark

In [10]:
# --- LazyPredict Benchmark ---
try:
    from lazypredict.Supervised import LazyRegressor
    n_lp = min(5000, len(X_train))
    lr = LazyRegressor(verbose=0, ignore_warnings=True)
    lp_models, _ = lr.fit(X_train.head(n_lp), X_test.head(min(1000, len(X_test))),
                          y_train.head(n_lp), y_test.head(min(1000, len(y_test))))
    print(lp_models.head(15))
except Exception as e:
    print(f"LazyPredict skipped: {e}")

                               Adjusted R-Squared  R-Squared      RMSE  \
Model                                                                    
PoissonRegressor                         0.157574   0.172753  5.695176   
LGBMRegressor                            0.149240   0.164569  5.723277   
HistGradientBoostingRegressor            0.123579   0.139370  5.808950   
SGDRegressor                             0.121100   0.136936  5.817161   
BayesianRidge                            0.119971   0.135827  5.820894   
LarsCV                                   0.118027   0.133918  5.827320   
ElasticNetCV                             0.117621   0.133519  5.828663   
RidgeCV                                  0.117214   0.133120  5.830006   
LassoCV                                  0.117132   0.133040  5.830276   
LassoLarsCV                              0.117120   0.133028  5.830314   
Ridge                                    0.116753   0.132667  5.831527   
Lars                                  

## FLAML AutoML

In [11]:
# --- FLAML AutoML ---
try:
    from flaml import AutoML
    automl = AutoML()
    automl.fit(X_train, y_train, task='regression', time_budget=60, seed=SEED, verbose=0)
    flaml_pred = automl.predict(X_test)
    print(f"FLAML best: {automl.best_estimator}")
    print(f"  R2={r2_score(y_test, flaml_pred):.4f}  RMSE={root_mean_squared_error(y_test, flaml_pred):.4f}  MAE={mean_absolute_error(y_test, flaml_pred):.4f}")
except Exception as e:
    print(f"FLAML skipped: {e}")

FLAML skipped: XGBModel.fit() got an unexpected keyword argument 'callbacks'


## Additional Models: CatBoost, LightGBM, XGBoost

In [12]:
# --- Boosting Models ---
models = {}
# CatBoost
cb = CatBoostRegressor(iterations=500, learning_rate=0.05, depth=6, random_seed=SEED, verbose=0)
cb.fit(X_train, y_train)
models['CatBoost'] = cb

# LightGBM
lgb = LGBMRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbose=-1)
lgb.fit(X_train, y_train)
models['LightGBM'] = lgb

# XGBoost
xgb = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=SEED, verbosity=0)
xgb.fit(X_train, y_train)
models['XGBoost'] = xgb

for name, m in models.items():
    p = m.predict(X_test)
    print(f"{name}: R2={r2_score(y_test, p):.4f}  RMSE={root_mean_squared_error(y_test, p):.4f}  MAE={mean_absolute_error(y_test, p):.4f}")

CatBoost: R2=0.3535  RMSE=6.2970  MAE=2.4962
LightGBM: R2=0.3194  RMSE=6.4608  MAE=2.4957
XGBoost: R2=0.3270  RMSE=6.4249  MAE=2.4936


## Top 3 Model Selection

In [13]:
# --- Top 3 Selection ---
all_results = {}
for name, m in models.items():
    p = m.predict(X_test)
    all_results[name] = {'R2': r2_score(y_test, p), 'RMSE': root_mean_squared_error(y_test, p), 'MAE': mean_absolute_error(y_test, p)}

results_df = pd.DataFrame(all_results).T.sort_values('R2', ascending=False)
print(results_df)
top3_names = results_df.head(3).index.tolist()
print(f'\nTop 3: {top3_names}')

                R2      RMSE       MAE
CatBoost  0.353518  6.297018  2.496164
XGBoost   0.327001  6.424863  2.493590
LightGBM  0.319445  6.460828  2.495673

Top 3: ['CatBoost', 'XGBoost', 'LightGBM']


## Final Evaluation of Top 3

In [14]:
# --- Final Evaluation of Top 3 ---
final_results = {}
for name in top3_names:
    m = models[name]
    p = m.predict(X_test)
    r2 = r2_score(y_test, p)
    rmse = root_mean_squared_error(y_test, p)
    mae = mean_absolute_error(y_test, p)
    final_results[name] = {'R2': r2, 'RMSE': rmse, 'MAE': mae}
    print(f"{name}: R2={r2:.4f}  RMSE={rmse:.4f}  MAE={mae:.4f}")

# Plot
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
names = list(final_results.keys())
for i, metric in enumerate(['R2', 'RMSE', 'MAE']):
    vals = [final_results[n][metric] for n in names]
    axes[i].bar(names, vals, color=['#2ecc71','#3498db','#e74c3c'])
    axes[i].set_title(metric)
    axes[i].tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'top3_comparison.png'), dpi=100)
plt.show()

CatBoost: R2=0.3535  RMSE=6.2970  MAE=2.4962
XGBoost: R2=0.3270  RMSE=6.4249  MAE=2.4936
LightGBM: R2=0.3194  RMSE=6.4608  MAE=2.4957


## Error Analysis

In [15]:
# --- Error Analysis ---
best_name = top3_names[0]
best_model = models[best_name]
preds = best_model.predict(X_test)
residuals = y_test - preds

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(residuals, bins=50, edgecolor='black')
axes[0].set_title('Residual Distribution')
axes[0].axvline(0, color='red', linestyle='--')
axes[1].scatter(preds, residuals, alpha=0.3, s=5)
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_title('Residuals vs Predicted')
axes[2].scatter(y_test, preds, alpha=0.3, s=5)
mn, mx = y_test.min(), y_test.max()
axes[2].plot([mn, mx], [mn, mx], 'r--')
axes[2].set_title('Actual vs Predicted')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, 'error_analysis.png'), dpi=100)
plt.show()

## Interpretation & Business Insight
Humidity and cloud cover drive rainfall predictions. Models can aid farmers in planning irrigation.

## Limitations
- Heavy zero-inflation skews metrics
- No temporal ordering used
- Location effects suppressed

## How to Improve
- Two-stage model: classify rain/no-rain, then predict amount
- Time series approach
- Separate models per climate zone

## Production Considerations
- Integrate with real-time BOM data feeds
- Automate retraining weekly
- Calibrate predictions for local conditions

## Common Mistakes
- Leaking RainTomorrow into features
- Not handling zero-inflated distribution
- Using R2 alone (misleading with zero-inflation)

## Mini Challenge
1. Build a two-stage model (classify then regress)
2. Extract month from Date for seasonality
3. Compare RMSE excluding zero-rainfall days

## Final Summary
Predicted rainfall amounts from weather data. Heavy zero-inflation makes this a challenging regression task.

In [16]:
# --- Save Metrics ---
metrics = {name: {k: round(v, 4) for k, v in vals.items()} for name, vals in final_results.items()}
metrics['best_model'] = top3_names[0]
with open(os.path.join(SAVE_DIR, 'metrics.json'), 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved metrics.json')
print(json.dumps(metrics, indent=2))

Saved metrics.json
{
  "CatBoost": {
    "R2": 0.3535,
    "RMSE": 6.297,
    "MAE": 2.4962
  },
  "XGBoost": {
    "R2": 0.327,
    "RMSE": 6.4249,
    "MAE": 2.4936
  },
  "LightGBM": {
    "R2": 0.3194,
    "RMSE": 6.4608,
    "MAE": 2.4957
  },
  "best_model": "CatBoost"
}
